In [28]:
import re
import csv

# مسیرها
input_path = r"F:\Thesis\project\2-RAG\raw_laws\Criminal Procedure Code\ehteram-azadi-hoghoogh-shahrvandi.txt"
output_tsv = r"F:\Thesis\project\2-RAG\raw_laws\Criminal Procedure Code\preprocessed_data\ehteram-azadi-hoghoogh-shahrvandi_articles.tsv"

LAW_CODE = "ehteraam_azadi_hoghoogh_shahrvandi"
LAW_NAME = "قانون احترام به آزادی های مشروع و حفظ حقوق شهروندی"


def normalize_digits(text: str) -> str:
    persian_digits = "۰۱۲۳۴۵۶۷۸۹"
    english_digits = "0123456789"
    return text.translate(str.maketrans(persian_digits, english_digits))


def normalize_persian(text: str) -> str:
    text = text.replace("ي", "ی").replace("ك", "ک")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# 1) خواندن
with open(input_path, "r", encoding="utf-8") as f:
    raw = f.read()

raw = normalize_digits(raw)
raw = raw.replace("–", "-").replace("—", "-").replace("ـ", "-")

# 2) شکستن خطوط برای مواد (با فاصله اختیاری بین ماده و عدد)
patterns_to_break = [
    r"((?:ماده|مواد)\s*‌?\s*\d+(?:تا\d+)?(?:\s*و\s*\d+)?(?:مکرر)?(?:\([^)]+\))?\s*[-–ـ:.][^\n]*)"
]

for p in patterns_to_break:
    raw = re.sub(r"(?<!^)" + p, r"\n\1", raw, flags=re.MULTILINE | re.UNICODE)

lines = raw.splitlines()

# 3) الگوی جامع ماده (با فاصله/نیم‌فاصله اختیاری)
article_pattern = re.compile(
    r"^\s*(?:ماده|مواد)"           # ماده یا مواد
    r"(?:\s|‌)*"                     # فاصله یا نیم‌فاصله (صفر یا بیشتر)
    r"("                              # شروع گروه شماره
        r"\d+"
        r"(?:\s*تا\s*\d+)?"
        r"(?:\s*و\s*\d+)*"
    r")"
    r"(?:\s*مکرر)?"
    r"(?:\s*\([^)]+\))?"
    r"\s*"
    r"[-–ـ:.]"                        # خط تیره، دونقطه، یا نقطه
    r"\s*(.*)$",
    re.UNICODE
)

articles = []
current_article_num = None
current_article_text = ""


def flush_article():
    global current_article_num, current_article_text
    if current_article_num is not None and current_article_text.strip():
        articles.append(
            {
                "law_code": LAW_CODE,
                "law_name": LAW_NAME,
                "article_number": current_article_num,
                "text": normalize_persian(current_article_text),
            }
        )
    current_article_num = None
    current_article_text = ""


# پردازش
for line in lines:
    stripped = line.strip()
    if not stripped:
        continue

    # تشخیص ماده
    m = article_pattern.match(stripped)
    if m:
        flush_article()
        num_str = m.group(1)
        rest = m.group(2).strip()

        # استخراج اولین عدد
        first_num = re.search(r"\d+", num_str)
        if first_num:
            try:
                current_article_num = int(first_num.group())
            except ValueError:
                current_article_num = None
        else:
            current_article_num = None

        current_article_text = f"ماده {num_str}- {rest}"
        continue

    # ادامه متن ماده
    if current_article_num is not None:
        current_article_text += " " + stripped

# آخرین ماده
flush_article()

# خروجی TSV
fieldnames = ["law_code", "law_name", "article_number", "text"]

with open(output_tsv, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames, delimiter="\t")
    writer.writeheader()
    for row in articles:
        writer.writerow(row)

print("✓ پردازش قانون نحوه اجرای محکومیت‌های مالی تمام شد.")
print(f"✓ تعداد مواد استخراج شده: {len(articles)}")
print(f"✓ خروجی TSV: {output_tsv}")


✓ پردازش قانون نحوه اجرای محکومیت‌های مالی تمام شد.
✓ تعداد مواد استخراج شده: 1
✓ خروجی TSV: F:\Thesis\project\2-RAG\raw_laws\Criminal Procedure Code\preprocessed_data\ehteram-azadi-hoghoogh-shahrvandi_articles.tsv
